# AI-Powered IT Incident Prediction & Prevention System

## Objective
To analyze server performance metrics and build a machine learning model that predicts whether an IT incident will occur.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("IT_Incident_Prediction_1000_Records.csv")

In [3]:
df

,incident_id,timestamp,server_id,department,environment,server_type,cpu_usage_pct,memory_usage_pct,disk_usage_pct,network_latency_ms,...,uptime_days,previous_incidents_30d,maintenance_due,anomaly_score,incident_occurred,severity,incident_type,status,resolution_time_minutes,root_cause
0,INC-0001,2025-01-01 07:05:00,SRV-008,Customer Support,Staging,Web Server,23.32,55.73,66.22,103.10,...,1,0,No,0.00,No,NaN,NaN,Normal,NaN,NaN
1,INC-0002,2025-01-01 09:59:00,SRV-010,Customer Support,Production,File Server,82.30,72.72,59.27,138.25,...,141,1,No,5.61,No,NaN,NaN,Normal,NaN,NaN
2,INC-0003,2025-01-01 20:06:00,SRV-043,Operations,Production,API Gateway,74.19,64.79,54.89,165.19,...,25,2,Yes,49.50,No,NaN,NaN,Normal,NaN,NaN
3,INC-0004,2025-01-02 05:33:00,SRV-050,Operations,Development,Database Server,22.94,54.27,42.49,46.62,...,60,1,Yes,6.83,No,NaN,NaN,Normal,NaN,NaN
4,INC-0005,2025-01-02 13:40:00,SRV-024,IT,Production,Application Server,48.15,39.90,79.66,65.56,...,41,0,No,11.10,No,NaN,NaN,Normal,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,INC-0996,2025-12-29 21:44:00,SRV-047,IT,Production,Application Server,42.36,44.77,55.41,11.90,...,40,1,No,0.81,No,NaN,NaN,Normal,NaN,NaN
996,INC-0997,2025-12-29 23:47:00,SRV-028,Sales,Development,Application Server,34.11,57.20,60.06,59.07,...,68,0,No,0.00,No,NaN,NaN,Normal,NaN,NaN
997,INC-0998,2025-12-30 12:41:00,SRV-047,Finance,Production,API Gateway,51.83,58.62,64.26,42.70,...,72,3,No,1.26,No,NaN,NaN,Normal,NaN,NaN
998,INC-0999,2025-12-31 10:23:00,SRV-012,Customer Support,Development,Database Server,56.29,30.38,75.21,30.26,...,138,0,No,9.28,No,NaN,NaN,Normal,NaN,NaN


In [4]:
df.shape

(1000, 25)

In [5]:
df.dtypes

incident_id                    str
timestamp                      str
server_id                      str
department                     str
environment                    str
server_type                    str
cpu_usage_pct              float64
memory_usage_pct           float64
disk_usage_pct             float64
network_latency_ms         float64
response_time_ms           float64
error_count                  int64
active_users                 int64
packet_loss_pct            float64
temperature_c              float64
uptime_days                  int64
previous_incidents_30d       int64
maintenance_due                str
anomaly_score              float64
incident_occurred              str
severity                       str
incident_type                  str
status                         str
resolution_time_minutes    float64
root_cause                     str
dtype: object

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   incident_id              1000 non-null   str    
 1   timestamp                1000 non-null   str    
 2   server_id                1000 non-null   str    
 3   department               1000 non-null   str    
 4   environment              1000 non-null   str    
 5   server_type              1000 non-null   str    
 6   cpu_usage_pct            1000 non-null   float64
 7   memory_usage_pct         1000 non-null   float64
 8   disk_usage_pct           1000 non-null   float64
 9   network_latency_ms       1000 non-null   float64
 10  response_time_ms         1000 non-null   float64
 11  error_count              1000 non-null   int64  
 12  active_users             1000 non-null   int64  
 13  packet_loss_pct          1000 non-null   float64
 14  temperature_c            1000 non-nu

In [7]:
df.isnull().sum()

incident_id                  0
timestamp                    0
server_id                    0
department                   0
environment                  0
server_type                  0
cpu_usage_pct                0
memory_usage_pct             0
disk_usage_pct               0
network_latency_ms           0
response_time_ms             0
error_count                  0
active_users                 0
packet_loss_pct              0
temperature_c                0
uptime_days                  0
previous_incidents_30d       0
maintenance_due              0
anomaly_score                0
incident_occurred            0
severity                   902
incident_type              902
status                       0
resolution_time_minutes    907
root_cause                 902
dtype: int64

In [8]:
df.duplicated().sum()


np.int64(0)

In [9]:
incident_columns = [
    "severity",
    "incident_type",
    "root_cause"
]

df[incident_columns] = df[incident_columns].fillna("No Incident")

In [10]:
df[incident_columns].isnull().sum()

severity         0
incident_type    0
root_cause       0
dtype: int64

In [11]:
df.isnull().sum()

incident_id                  0
timestamp                    0
server_id                    0
department                   0
environment                  0
server_type                  0
cpu_usage_pct                0
memory_usage_pct             0
disk_usage_pct               0
network_latency_ms           0
response_time_ms             0
error_count                  0
active_users                 0
packet_loss_pct              0
temperature_c                0
uptime_days                  0
previous_incidents_30d       0
maintenance_due              0
anomaly_score                0
incident_occurred            0
severity                     0
incident_type                0
status                       0
resolution_time_minutes    907
root_cause                   0
dtype: int64

In [12]:
df[df["resolution_time_minutes"].isnull()][
    ["incident_occurred", "status"]
].value_counts()

incident_occurred  status
No                 Normal    902
Yes                Open        5
Name: count, dtype: int64

In [13]:
df.loc[
    df["incident_occurred"] == "No",
    "resolution_time_minutes"
] = 0

In [14]:
df.isnull().sum()

incident_id                0
timestamp                  0
server_id                  0
department                 0
environment                0
server_type                0
cpu_usage_pct              0
memory_usage_pct           0
disk_usage_pct             0
network_latency_ms         0
response_time_ms           0
error_count                0
active_users               0
packet_loss_pct            0
temperature_c              0
uptime_days                0
previous_incidents_30d     0
maintenance_due            0
anomaly_score              0
incident_occurred          0
severity                   0
incident_type              0
status                     0
resolution_time_minutes    5
root_cause                 0
dtype: int64

In [15]:
df[df["resolution_time_minutes"].isnull()][
    ["incident_id", "incident_occurred", "status"]
]

,incident_id,incident_occurred,status
56,INC-0057,Yes,Open
232,INC-0233,Yes,Open
269,INC-0270,Yes,Open
325,INC-0326,Yes,Open
644,INC-0645,Yes,Open


In [16]:
df.loc[
    (df["incident_occurred"] == "Yes") &
    (df["status"] == "Open"),
    "resolution_time_minutes"
] = -1

In [17]:
df.isnull().sum().sum()

np.int64(0)

In [18]:
df.isnull().sum()

incident_id                0
timestamp                  0
server_id                  0
department                 0
environment                0
server_type                0
cpu_usage_pct              0
memory_usage_pct           0
disk_usage_pct             0
network_latency_ms         0
response_time_ms           0
error_count                0
active_users               0
packet_loss_pct            0
temperature_c              0
uptime_days                0
previous_incidents_30d     0
maintenance_due            0
anomaly_score              0
incident_occurred          0
severity                   0
incident_type              0
status                     0
resolution_time_minutes    0
root_cause                 0
dtype: int64

In [19]:
df

,incident_id,timestamp,server_id,department,environment,server_type,cpu_usage_pct,memory_usage_pct,disk_usage_pct,network_latency_ms,...,uptime_days,previous_incidents_30d,maintenance_due,anomaly_score,incident_occurred,severity,incident_type,status,resolution_time_minutes,root_cause
0,INC-0001,2025-01-01 07:05:00,SRV-008,Customer Support,Staging,Web Server,23.32,55.73,66.22,103.10,...,1,0,No,0.00,No,No Incident,No Incident,Normal,0.0,No Incident
1,INC-0002,2025-01-01 09:59:00,SRV-010,Customer Support,Production,File Server,82.30,72.72,59.27,138.25,...,141,1,No,5.61,No,No Incident,No Incident,Normal,0.0,No Incident
2,INC-0003,2025-01-01 20:06:00,SRV-043,Operations,Production,API Gateway,74.19,64.79,54.89,165.19,...,25,2,Yes,49.50,No,No Incident,No Incident,Normal,0.0,No Incident
3,INC-0004,2025-01-02 05:33:00,SRV-050,Operations,Development,Database Server,22.94,54.27,42.49,46.62,...,60,1,Yes,6.83,No,No Incident,No Incident,Normal,0.0,No Incident
4,INC-0005,2025-01-02 13:40:00,SRV-024,IT,Production,Application Server,48.15,39.90,79.66,65.56,...,41,0,No,11.10,No,No Incident,No Incident,Normal,0.0,No Incident
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,INC-0996,2025-12-29 21:44:00,SRV-047,IT,Production,Application Server,42.36,44.77,55.41,11.90,...,40,1,No,0.81,No,No Incident,No Incident,Normal,0.0,No Incident
996,INC-0997,2025-12-29 23:47:00,SRV-028,Sales,Development,Application Server,34.11,57.20,60.06,59.07,...,68,0,No,0.00,No,No Incident,No Incident,Normal,0.0,No Incident
997,INC-0998,2025-12-30 12:41:00,SRV-047,Finance,Production,API Gateway,51.83,58.62,64.26,42.70,...,72,3,No,1.26,No,No Incident,No Incident,Normal,0.0,No Incident
998,INC-0999,2025-12-31 10:23:00,SRV-012,Customer Support,Development,Database Server,56.29,30.38,75.21,30.26,...,138,0,No,9.28,No,No Incident,No Incident,Normal,0.0,No Incident


In [20]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
cpu_usage_pct,1000.0,52.56603,17.540522,9.63,41.2500,49.885,61.3225,100.00
memory_usage_pct,1000.0,53.81774,13.262403,19.41,45.0175,52.170,60.0850,99.13
disk_usage_pct,1000.0,63.31690,15.747513,12.00,52.7575,62.615,74.3675,100.00
network_latency_ms,1000.0,60.09404,51.669733,2.00,28.9750,43.975,68.3025,267.57
response_time_ms,1000.0,392.77377,343.615441,60.00,199.4925,281.490,399.7800,1746.97
error_count,1000.0,3.46000,4.758891,0.00,0.0000,2.000,5.0000,25.00
active_users,1000.0,228.38300,102.346789,5.00,151.0000,224.500,302.0000,531.00
packet_loss_pct,1000.0,1.20323,1.492938,0.00,0.2600,0.585,1.4100,7.65
temperature_c,1000.0,56.26227,9.281043,32.72,50.1050,55.290,60.9425,86.74
uptime_days,1000.0,88.54000,52.589410,1.00,49.0000,87.000,124.0000,260.00


In [21]:
df["incident_occurred"].value_counts()

incident_occurred
No     902
Yes     98
Name: count, dtype: int64

In [22]:
target_percentage = (
    df["incident_occurred"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(target_percentage)

incident_occurred
No     90.2
Yes     9.8
Name: proportion, dtype: float64


# Interpretation
Around 90.2% of records are normal.
Around 9.8% contain an IT incident.
The dataset is imbalanced because normal records are much higher than incident records.
Later, we will use appropriate evaluation metrics such as precision, recall, F1-score and ROC-AUC.

In [23]:
df["severity"].value_counts()

severity
No Incident    902
Low             54
Critical        28
Medium           9
High             7
Name: count, dtype: int64

In [24]:
df["incident_type"].value_counts()

incident_type
No Incident             902
Network Latency          34
CPU Overload             26
Application Error        26
Memory Leak               5
Disk Capacity             4
Hardware Overheating      3
Name: count, dtype: int64

In [25]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])
print("Duplicate records:", df.duplicated().sum())
print("Total incidents:", (df["incident_occurred"] == "Yes").sum())
print("Normal records:", (df["incident_occurred"] == "No").sum())
print("Incident rate:", round((df["incident_occurred"] == "Yes").mean() * 100, 2), "%")

DATASET SUMMARY
Total records: 1000
Total columns: 25
Duplicate records: 0
Total incidents: 98
Normal records: 902
Incident rate: 9.8 %


## Phase 1 Conclusion

The dataset contains 1,000 IT system monitoring records and 25 columns.

The target variable is `incident_occurred`, which indicates whether an IT incident occurred.

The dataset contains system performance metrics such as CPU usage, memory usage, disk usage, network latency, response time, error count, temperature, and anomaly score.

Approximately 9.8% of the records contain incidents, indicating an imbalanced classification problem.

In [28]:
# Save cleaned data

df.to_csv("Cleaned_IT_Incident_Data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
